In [4]:
import pandas as pd
from datetime import datetime

In [5]:
orderData = pd.read_csv('../../dataMyself/订单信息.csv', encoding='GBK')

In [6]:
class Patient:
    def __init__(self, index, age, sex):
        self.ID = ''
        self.index = index
        self.age = age
        self.sex = sex
        self.nurseList = []
        self.orderList = [index]

    def setID(self, ID):
        self.ID = ID

    def addNurse(self, ID):
        self.nurseList.append(ID)

    def addOrder(self, index):
        self.orderList.append(index)

    def toString(self):
        return f'{self.index}, {self.age}, {self.sex}, {self.nurseList}, {self.orderList}'

In [7]:
patientList = []

for hospital in orderData['医院名称'].unique().tolist():
    zoneList = orderData[orderData['医院名称'] == hospital]['病区名称'].unique().tolist()
    for zone in zoneList:
        df = orderData[orderData['病区名称'] == zone]
        bedList = df['床位号'].unique().tolist()
        for bed in bedList:
            visited = {}
            for i, row in df[df['床位号'] == bed].iterrows():
                patientAge, patientSex = row['患者年龄'], row['患者性别']
                # if patientAge == 0:
                #     continue
                patientInfo = f'{patientAge}_{patientSex}'
                if patientInfo not in visited.keys():
                    patient = Patient(i, patientAge, patientSex)
                    patient.addNurse(row['护工ID'])
                    visited[patientInfo] = [patient]
                else:
                    j = 0
                    patients = visited[patientInfo]
                    for p in patients:
                        endTime = datetime.strptime(orderData.iloc[p.index]['服务结束时间'].split(' ')[0], '%Y/%m/%d')
                        startTime = datetime.strptime(row['服务开始时间'].split(' ')[0], '%Y/%m/%d')
                        diffDays = (startTime - endTime).days
                        if diffDays <= 7:
                            p.addOrder(i)
                            if row['护工ID'] not in p.nurseList:
                                p.addNurse(row['护工ID'])
                            break
                        j += 1

                    if j == len(patients):
                        patient = Patient(i, patientAge, patientSex)
                        patient.addNurse(row['护工ID'])
                        visited[patientInfo].append(patient)
            for patients in visited.values():
                for patient in patients:
                    patientList.append(patient)

In [8]:
patientData = pd.DataFrame()

IDs = []
ages = []
sexes = []
for i, patient in enumerate(patientList):
    patientID = i + 1
    patient2Order = f'{patientID}->{patient.orderList}'
    patient2Nurse = f'{patientID}->{patient.nurseList}'

    patient.setID(patientID)

    IDs.append(patientID)
    ages.append(patient.age)
    sexes.append(patient.sex)

patientData['患者ID'] = IDs
patientData['性别'] = sexes
patientData['年龄'] = ages
patientData.to_csv('患者信息.csv', index=False)

In [9]:
with open('患者_订单.txt', 'w') as file:
    for patient in patientList:
        file.write(f'{patient.ID}->{patient.orderList}\n')

with open('患者_护工.txt', 'w') as file:
    for patient in patientList:
        file.write(f'{patient.ID}->{patient.nurseList}\n')

In [10]:
with open('护工_患者.txt', 'w') as file:
    for nurseID in orderData['护工ID'].unique().tolist():
        def filterFunc(item):
            return nurseID in item.nurseList

        tmp = [item.ID for item in filter(filterFunc, patientList)]
        file.write(f'{nurseID}->{tmp}\n')